# Suite LAY — Layers' interpreters: the base class, and a layer not built on proxies

Each layer of a stack interprets a transaction with an interpreter of its own, an instance of a class coupled with the
layer, which reads and writes the layer directly, not through the Store interface
([the stacked stores proposal, stacking by mutation transactions](https://github.com/pitaman71/mbse-basis/blob/main/proposals/STACKED-STORES.md#stacking-by-mutation-transactions)). The base class,
`Transactions.Interpreter`, holds what every layer interprets alike: the walk of a transaction's terms (map, upsert,
update, reduce, partition, then), binding symbols, frames, errors annotated on frames
([CAP-309: errors annotated on frames](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/README.md#CAP-309)), unique clauses checked at commit, and the stream `txn.exec` gives, with its
scopes, remainders, completion markers and journals. A kind of layer's class gives it what only that layer knows: its
reads of its own storage (the candidates for a symbol's schema, the object a reference designates, the entries that
link an object), its writes, and its transaction's savepoints. A store in memory's is `Transactions.InMemory`.

This suite's layer, `Indexed`, holds its data in plain dictionaries, as an index would: each object a row, by an id of
its own, found by its reference's path through a dictionary keyed by the path; each relation's entries a list of rows
linking objects by their ids. It is the test's, not shipped, and interprets with the base class alone, over the schemas
of a `Builders.Store`, so that the same transactions as suites TXN and EXE give the same frames, by their references,
and leave the same data as a store in memory does.

TypeScript and Python run this suite with the same case IDs. The case setup is below.

In [ ]:
import { Builders as B, JSON as Json, Plain, Proxies, Schemas as S, Transactions } from "@mbse/schemas/Framework";
import { LookupError, ValueError } from "@mbse/schemas/Framework/Errors";
import { assert, equal, map, raises } from "./support.js";

const { Frame } = Transactions;

type Row = { id: number; schema: string; path: any; values: Map<string, unknown> };
type Entry = { links: Map<string, number>; values: Map<string, unknown> };

/** A layer kept in plain maps, as an index would keep it, interpreted by the base class alone: each object a row,
 * `{id, schema, path, values}`, its properties' values plain, by an id of its own; `paths`, each object that has a
 * reference, by its path's JSON; and each relation's entries, by its name, a list of rows `{links, values}`, linking
 * objects by their ids. Its transaction keeps a log of how to undo each write. */
class Indexed extends Transactions.Interpreter {
  readonly objects = new Map<number, Row>();
  readonly paths = new Map<string, number>();
  readonly relations = new Map<string, Entry[]>();
  readonly counted = new Map<string, Row>();
  ids = 0;
  undo: (() => void)[] | null = null; // the transaction open's undo log; null where none is open
  inserted: [any, Entry][] = [];
  missed = new Set<string>();
  txnUid = "";

  constructor(schemas: any) {
    super(schemas);
    for (const name of schemas.names()) { // each singleton, held from the start
      const schema = schemas.registered(name);
      if (schema instanceof S.OfObject.Data && schema.singleton !== null) {
        this.row(name, map({ singleton: schema.singleton, next: { target: {} } }), new Map(), false);
      }
    }
  }

  // --- its storage ---

  row(schemaName: string, path: any = null, values = new Map<string, unknown>(), logged = true): Row {
    const row: Row = { id: this.ids++, schema: schemaName, path: null, values: new Map(values) };
    this.objects.set(row.id, row);
    if (logged) this.undo!.push(() => this.objects.delete(row.id));
    if (path !== null) this.place(row, path, logged);
    return row;
  }

  place(row: Row, path: any, logged = true): void {
    const key = Json.dumps(path);
    row.path = path;
    this.paths.set(key, row.id);
    if (logged) this.undo!.push(() => { this.paths.delete(key); row.path = null; });
  }

  held(relation: any): Entry[] {
    if (!this.relations.has(relation.name)) this.relations.set(relation.name, []);
    return this.relations.get(relation.name)!;
  }

  // --- its reads ---

  candidates(schema: any): any[] {
    const name = this.store.name_of(schema);
    return [...this.objects.values()].filter((row) => row.schema === name);
  }

  named(): any[] {
    return []; // it holds no schema objects: its schemas are its store's
  }

  designated(path: any): any {
    const found = this.paths.get(Json.dumps(path));
    return found === undefined ? null : this.objects.get(found) ?? null;
  }

  schema_of(target: Row): any {
    return this.store.registered(target.schema);
  }

  key(target: Row): unknown {
    return target.id;
  }

  value(holder: Row | Entry, _schema: any, name: string): [boolean, unknown] {
    return holder.values.has(name) ? [true, holder.values.get(name)] : [false, null];
  }

  entries(target: Row, schema: any, name: string): any[] {
    const adjacency = schema.adjacencies.get(name);
    return this.held(adjacency.relation).filter((entry) => entry.links.get(adjacency.me) === target.id);
  }

  link(entry: Entry, name: string): any {
    const id = entry.links.get(name);
    return id === undefined ? null : this.objects.get(id) ?? null;
  }

  linking(target: Row, relation: any): any[] {
    return this.held(relation).filter((entry) => [...entry.links.values()].includes(target.id));
  }

  part(entry: Entry, name: string): unknown {
    return entry.values.has(name) ? Json.dumps(entry.values.get(name) as any) : null;
  }

  shown(entry: Entry, names: string[]): string {
    const one = (name: string): string => {
      const target = this.link(entry, name);
      return target !== null ? `<${target.schema}>` : `'${entry.values.get(name)}'`;
    };
    return `{${names.map((name) => `${name}: ${one(name)}`).join(", ")}}`;
  }

  given(target: any, names: readonly string[]): [any, Map<string, unknown>] {
    if (!(target instanceof Object) || !("schema" in target) || typeof target.schema !== "string") {
      return [target, new Map()]; // a schema object, its store's
    }
    if (target.path === null) {
      return [new Transactions._Refused(`a counted '${target.schema}' has no reference`, target, this.life), new Map()];
    }
    const reference = Plain.FromPlain(this.store)(this.store.registered("References.OfObject"), map({ path: {} })
      .set("path", target.path));
    return [reference, new Map(names.filter((name) => target.values.has(name)).map((name) => [name, target.values.get(name)]))];
  }

  // --- its writes, each logged to be undone ---

  assign(target: Row, _schema: any, name: string, verb: Map<string, any>): void {
    const had = new Map(target.values);
    if (verb.has("set")) target.values.set(name, verb.get("set").get("value"));
    else target.values.delete(name);
    this.undo!.push(() => { target.values.clear(); for (const [k, v] of had) target.values.set(k, v); });
  }

  insert(target: Row, schema: any, name: string, links: Map<string, Row>, values: Map<string, unknown>): void {
    const adjacency = schema.adjacencies.get(name);
    const relation = adjacency.relation;
    const held = this.held(relation);
    if (Proxies._single(relation, adjacency.me) && !this.missed.has(relation.name)) {
      this.remove(target, schema, name, this.entries(target, schema, name).map((_, index) => index));
    }
    const entry: Entry = { links: new Map([[adjacency.me, target.id], ...[...links].map(([link, row]) => [link, row.id] as [string, number])]),
      values: new Map(values) };
    held.push(entry);
    this.inserted.push([relation, entry]);
    this.undo!.push(() => held.splice(held.indexOf(entry), 1));
    for (const [link, row] of links) { // an object it owns is placed: its path its owner's, one step on
      const owned = [...this.schema_of(row).adjacencies.values()].filter((a: any) => a.owner && a.relation === relation
        && a.me === link);
      if (owned.length > 0 && target.path !== null) {
        const step = map({ entry: { i: name, o: owned[0].name, properties: {}, next: { target: {} } } });
        step.get("entry").set("properties", new Map(values));
        this.place(row, new Map([["singleton", target.path.get("singleton")], ["next", deeper(target.path.get("next"), step)]]));
      }
    }
  }

  remove(target: Row, schema: any, name: string, positions: number[]): void {
    const relation = schema.adjacencies.get(name).relation;
    if (positions.length === 0) {
      this.missed.add(relation.name);
      return;
    }
    const held = this.held(relation);
    const mine = this.entries(target, schema, name);
    for (const index of [...positions].sort((a, b) => b - a)) {
      const entry = mine[index];
      const at = held.indexOf(entry);
      held.splice(at, 1);
      this.undo!.push(() => held.splice(at, 0, entry));
    }
  }

  create(schema: any, values: Map<string, Map<string, any>>): any {
    return this.row(this.store.name_of(schema), null,
      new Map([...values].map(([name, verb]) => [name, verb.get("set").get("value")])));
  }

  upserted(index: string, made: any = null): any {
    if (made !== null) this.counted.set(index, made);
    return this.counted.get(index) ?? null;
  }

  // --- its transaction ---

  savepoint<T>(body: () => T): T {
    const mark = this.undo!.length;
    try {
      return body(); // it gives no journal
    } catch (error) {
      this.rollback(mark);
      throw error;
    }
  }

  rollback(mark: number): void {
    while (this.undo!.length > mark) this.undo!.pop()!();
  }

  uid(): string {
    return this.txnUid;
  }

  /** The layer's transaction: `body` given `txn`, whose `exec` this layer's interpreter runs; at its end, the unique
   * clauses of what it inserted are checked, and it commits, or, where it throws, every write is undone. */
  transaction<T>(body: (txn: any) => T, uid = "t/1"): T {
    this.undo = [];
    this.inserted = [];
    this.missed = new Set();
    this.txnUid = uid;
    const life = { ended: false };
    const txn = new Transactions.Transaction(this, life);
    try {
      const result = body(txn);
      txn._raise();
      const byRelation = new Map<string, [any, Entry[]]>();
      for (const [relation, entry] of this.inserted) {
        if (relation.uniques.length === 0 || !this.held(relation).includes(entry)) continue;
        if (!byRelation.has(relation.name)) byRelation.set(relation.name, [relation, []]);
        byRelation.get(relation.name)![1].push(entry);
      }
      for (const [relation, entries] of byRelation.values()) this._check_uniques(relation, entries);
      return result;
    } catch (error) {
      this.rollback(0);
      throw error;
    } finally {
      life.ended = true;
      this.undo = null;
    }
  }
}

/** A path's `next`, its innermost `{"target": {}}` given a step more. */
function deeper(nested: Map<string, any>, step: Map<string, any>): Map<string, any> {
  if (nested.has("target")) return step;
  const entry = new Map(nested.get("entry"));
  entry.set("next", deeper(entry.get("next"), step));
  return new Map([["entry", entry]]);
}

/** crm.Person, owned by the singleton crm.TheBook under a key, who knows persons; crm.Note, counted. */
function crm(txn: any): any {
  const S = txn.Schemas;
  return txn.uid("crm/1").mutations([(mu: any) => mu.upsert((u: any) => u.exists([(p: any) => p
    .symbols({ Knows: S.OfRelation, Person: S.OfObject, Holds: S.OfRelation, Book: S.OfObject, Note: S.OfObject })
    .spec([
      (_: any) => _.Knows.name("crm.Knows").links([(t: any) => t.name("knower"), (t: any) => t.name("known")]),
      (_: any) => _.Person.name("crm.Person").ref(true).properties({ name: S.OfStr, age: S.OfInt }).adjacencies([
        (t: any) => t.name("knows").relation([_.Knows]).me("knower"),
        (t: any) => t.relation([_.Knows]).me("known"),
        (t: any) => t.name("book").relation([_.Holds]).me("contact").owner(true),
      ]),
      (_: any) => _.Holds.name("crm.Holds").links([(t: any) => t.name("book"), (t: any) => t.name("contact")]).properties({
        key: S.OfStr }).uniques([["book", "key"], ["contact"]]),
      (_: any) => _.Book.name("crm.Book").ref(true).singleton("crm.TheBook").adjacencies({
        contacts: (t: any) => t.relation([_.Holds]).me("book") }),
      (_: any) => _.Note.name("crm.Note").ref(true).properties({ text: S.OfStr }),
    ])]))]);
}

/** Ada, 36, who knows Bob, and Bob, of no age: crm.TheBook's contacts "ada" and "bob". */
function peopleOf(t: any): any {
  return t.uid("people/1").mutations([(mu: any) => mu.upsert((u: any) => u.exists([(p: any) => p.symbols(
    { a: t.crm.Person, b: t.crm.Person, book: t.crm.Book }).spec([
    (_: any) => _.book.contacts({ ada: (e: any) => e.contact(_.a), bob: (e: any) => e.contact(_.b) }),
    (_: any) => _.a.name("Ada").age(36n).knows([_.b]),
    (_: any) => _.b.name("Bob"),
  ])]))]);
}

/** A stream's items, collected as they are given; its error thrown. */
function items(stream: any): any[] {
  const given: any[] = [];
  let failure: { error: unknown } | null = null;
  stream.subscribe({ next: (item: unknown) => given.push(item), error: (error: unknown) => { failure = { error }; } });
  if (failure !== null) throw (failure as { error: unknown }).error;
  return given;
}

/** A transaction's expression, built over the store's schemas. */
function expression(store: any, spec: unknown): any {
  return store._expressed(spec);
}

/** The items of `txn.exec`'s stream in a layer of plain maps, within a transaction of their own. */
function run(layer: Indexed, frame: unknown, spec: unknown, allowWrite: boolean): any[] {
  return layer.transaction((txn: any) => items(txn.exec(frame, expression(layer.store, spec), allowWrite)));
}

/** A store in memory holding crm's schemas, Ada and Bob; and a layer of plain maps over its schemas, given the same
 * people by its own interpreter. */
function pair(): [any, Indexed] {
  const store: any = new B.Store();
  store.transaction(crm);
  const layer = new Indexed(store);
  store.transaction(peopleOf);
  run(layer, new Frame(), peopleOf, true);
  return [store, layer];
}

/** The frames of a stream's items, its completion markers left out. */
function frames(given: any[]): any[] {
  return given.filter((item) => item instanceof Frame);
}

/** Each frame's bindings, by symbol, as their paths, and what it assigns. */
function paths(store: any, given: any[]): string[] {
  return given.map((frame) => Json.dumps(map({
    bound: Object.fromEntries([...frame].sort(([a], [b]) => a < b ? -1 : 1).map(([symbol, target]: [string, any]) =>
      [symbol, Json.dumps(Plain.ToPlain(store)(target.as_schema(), target).get("path"))])),
    assigns: {} }).set("assigns", new Map([...frame.assigns].map(([s, v]: [string, any]) => [s, new Map(v)])))));
}

/** Each person, the store's and the layer's: name, age and whom they know, by name. */
function people(store: any, layer: Indexed): [unknown[], unknown[]] {
  const order = (a: any, b: any): number => a[0] < b[0] ? -1 : 1;
  const held = store.transaction(() => [...store.extent("crm.Person")].map((p: any) => [p.name.get(),
    store.member(p, "age"), [...p.knows].map((e: any) => e.known.name.get()).sort()]).sort(order));
  const Person = store.registered("crm.Person");
  const kept = layer.candidates(Person).map((row: Row) => [row.values.get("name"), row.values.get("age") ?? null,
    layer.entries(row, Person, "knows").map((entry: Entry) => layer.link(entry, "known").values.get("name")).sort()])
    .sort(order);
  return [held, kept];
}

/** A transaction: a map binding Ada to x, then the term `then` gives. */
function adaThen(then: (Person: any) => unknown): (txn: any) => any {
  return (txn: any) => {
    const Person = txn.crm.Person;
    return txn.uid("ada/1").mutations([(mu: any) => mu.map((m: any) => m.symbols([
      (p: any) => p.symbols({ x: Person }).spec([(_: any) => _.x.name("Ada")])]).then([then(Person)]))]);
  };
}

/** An update writing x's age. */
function aged(age: bigint | null): (Person: any) => unknown {
  return (Person: any) => (w: any) => w.update((u: any) => u.apply([
    (p: any) => p.symbols({ x: Person }).spec([(_: any) => _.x.age(age)])]));
}

<a id="LAY-01"></a>
## LAY-01 · The base class and a layer's: `Transactions.Interpreter` holds the walk and asks a layer for its reads, writes and savepoints, each a method a layer's class must give, so that a class that gives none cannot be made; a store in memory's interpreter is `Transactions.InMemory`, one for the store, coupled with it, which its transactions' `exec` runs

In [ ]:
{
  raises(TypeError, () => new (Transactions.Interpreter as any)(new B.Store()), "abstract"); // what a layer gives, it must give
  const store: any = new B.Store();
  assert(store._interpreter() instanceof Transactions.InMemory && store._interpreter() === store._interpreter());
  assert(store._interpreter().store === store && !store._interpreter().cache); // no layer below: it holds all it holds
  store.transaction((txn: any = null) => {
    assert(txn._interpreter === store._interpreter());
  });
  assert((new B.Store(store) as any)._interpreter().cache); // a cache layer's
  const hooks = ["candidates", "named", "designated", "schema_of", "key", "value", "entries", "link", "linking", "part",
    "shown", "given", "assign", "insert", "remove", "create", "upserted", "savepoint", "uid"];
  assert(hooks.every((hook) => !(hook in Transactions.Interpreter.prototype) && hook in Transactions.InMemory.prototype));
}

<a id="LAY-02"></a>
## LAY-02 · A layer not built on proxies interprets as a store in memory does: its data plain dictionaries, its reads and writes its own, the base class drives an upsert that makes owned and counted objects, run again finding them; a map through entries and properties; an update; a reduce; a partition's branches joined; its frames, by their references, and its data, the same as a store in memory's

In [ ]:
{
  const [store, layer] = pair();
  assert(equal(people(store, layer), [[["Ada", 36n, ["Bob"]], ["Bob", null, []]], [["Ada", 36n, ["Bob"]], ["Bob", null, []]]]));
  assert(!layer.cache && [...layer.objects.values()].every((row) => layer.knows(row))); // it holds all it holds
  const contact = (key: string): string => Json.dumps(map({ singleton: "crm.TheBook", next: { entry: { i: "contacts",
    o: "book", properties: { key }, next: { target: {} } } } }));
  assert(equal([...layer.paths.keys()].sort(), [Json.dumps(map({ singleton: "crm.TheBook", next: { target: {} } })),
    contact("ada"), contact("bob")].sort()));
  const again = frames(run(layer, new Frame(), peopleOf, true)); // run again: it finds what it made, writing nothing
  assert(equal(paths(store, again), paths(store, store.transaction(peopleOf)))
    && layer.candidates(store.registered("crm.Person")).length === 2);
  const note = (t: any): any => t.uid("note/1").mutations([(mu: any) => mu.upsert((u: any) => u.exists([(p: any) => p
    .symbols({ n: t.crm.Note }).spec([(_: any) => _.n.text("x")])]))]);
  const [first] = frames(run(layer, new Frame(), note, true));
  raises(LookupError, () => first.get("n"), "has no reference"); // a counted object: refused, as a store in memory refuses it
  layer.transaction((txn: any) => { // the same transaction's uid: the same counted object, found by what made it
    items(txn.exec(new Frame(), expression(store, note), true));
    items(txn.exec(new Frame(), expression(store, note), true));
  }, "t/1");
  layer.transaction((txn: any) => items(txn.exec(new Frame(), expression(store, note), true)), "t/2"); // another's
  assert(equal(layer.candidates(store.registered("crm.Note")).map((row: Row) => row.values), [map({ text: "x" }),
    map({ text: "x" })]));
  const knowing = (t: any): any => t.uid("knowing/1").mutations([(mu: any) => mu.map((m: any) => m.symbols([(p: any) => p
    .symbols({ x: t.crm.Person, y: t.crm.Person }).spec([(_: any) => _.x.knows([_.y]), (_: any) => _.y.name("Bob")])]))]);
  for (const spec of [knowing, adaThen(aged(40n)), adaThen(aged(null))]) {
    assert(equal(paths(store, frames(run(layer, new Frame(), spec, true))), paths(store, store.transaction(spec))));
  }
  const [theirs, mine] = people(store, layer);
  assert(equal(theirs, mine) && equal(mine, [["Ada", null, ["Bob"]], ["Bob", null, []]]));
  const pairs = (t: any): any => t.uid("pairs/1").mutations([(mu: any) => mu.map((m: any) => m.symbols([(p: any) => p
    .symbols({ x: t.crm.Person, y: t.crm.Person })]).then([(w: any) => w.reduce((r: any) => r.symbols([
    (c: any) => c.symbols(["y"])]))]))]);
  const split = (t: any): any => t.uid("split/1").mutations([(mu: any) => mu.map((m: any) => m.symbols([(p: any) => p
    .symbols({ x: t.crm.Person })]).then([(w: any) => w.partition((q: any) => q.branches([
    (e: any) => e.branch((b: any) => b.then([(w2: any) => w2.map((m2: any) => m2.symbols([(p: any) => p.symbols(
      { x: t.crm.Person }).spec([(_: any) => _.x.name("Ada")])]))])),
    (e: any) => e.branch((b: any) => b.then([(w2: any) => w2.update((u: any) => u.apply([(p: any) => p.symbols(
      { x: t.crm.Person }).spec([(_: any) => _.x.age(7n)])]))])),
  ]))]))]);
  for (const spec of [pairs, split]) {
    const ours = paths(store, frames(run(layer, new Frame(), spec, true)));
    assert(equal(ours.sort(), paths(store, store.transaction(spec)).sort()) && ours.length === (spec === pairs ? 2 : 3));
  }
  const [held, kept] = people(store, layer);
  assert(equal(held, kept) && equal(kept, [["Ada", 7n, ["Bob"]], ["Bob", 7n, []]]));
}

<a id="LAY-03"></a>
## LAY-03 · A layer not built on proxies streams as a store in memory does: `txn.exec` over input frames binding symbols by their references, each frame carrying its input frame's id, then its completion marker; read-only, a frame halting where a term would write, given with its remainder, which the same transaction completes writing, the loopback; a partition's join kept through the loopback; an error annotated on its frame, the scope then raising it and undoing every write

In [ ]:
{
  const [store, layer] = pair();
  const everyone = (t: any): any => t.uid("everyone/1").mutations([(mu: any) => mu.map((m: any) => m.symbols([
    (p: any) => p.symbols({ x: t.crm.Person })]))]);
  const older = (t: any): any => t.uid("older/1").mutations([(mu: any) => mu.update((u: any) => u.apply([
    (p: any) => p.symbols({ x: t.crm.Person }).spec([(_: any) => _.x.age(50n)])]))]);
  const given = frames(run(layer, new Frame(), everyone, true));
  const each = run(layer, given.map((frame: any) => new Map([["x", frame.get("x")]])), older, true); // by their references
  assert(equal(each.map((item: any) => item instanceof Frame ? item.scope : item),
    ["0", map({ end: "0" }), "1", map({ end: "1" })]));
  assert(frames(each).every((frame: any) => equal(frame.assigns, map({ x: { age: 50n } }))));
  assert(equal(people(store, layer)[1], [["Ada", 50n, ["Bob"]], ["Bob", 50n, []]]));
  const looped = layer.transaction((txn: any) => {
    const [halted] = frames(items(txn.exec(new Frame(), expression(store, adaThen(aged(60n))), false))); // it would write
    assert(halted.remainder.subterm === "$.mutations[0].map.then[0].update" && equal(people(store, layer)[1][0],
      ["Ada", 50n, ["Bob"]]));
    return frames(items(txn.exec(halted, halted.remainder, true)))[0]; // the loopback: the same transaction, writing
  });
  assert(equal(looped.assigns, map({ x: { name: "Ada", age: 60n } })) && equal(people(store, layer)[1][0],
    ["Ada", 60n, ["Bob"]]));
  const split = (t: any): any => t.uid("split/2").mutations([(mu: any) => mu.map((m: any) => m.symbols([(p: any) => p
    .symbols({ x: t.crm.Person }).spec([(_: any) => _.x.name("Bob")])]).then([(w: any) => w.partition((q: any) => q.branches([
    (e: any) => e.branch((b: any) => b.then([(w2: any) => w2.update((u: any) => u.apply([(p: any) => p.symbols(
      { x: t.crm.Person }).spec([(_: any) => _.x.age(3n)])]))]))]).then([(w3: any) => w3.map((m3: any) => m3.symbols([
    (p: any) => p.symbols({ bk: t.crm.Book })]))]))]))]);
  const [joined] = frames(run(layer, new Frame(), split, false)); // read-only, it halts within the branch: the loopback
  assert(joined.remainder === null && equal([...joined.keys()].sort(), ["bk", "x"])
    && equal(people(store, layer)[1][1], ["Bob", 3n, []]));
  const nobody = map({ singleton: "crm.TheBook", next: { entry: { i: "contacts", o: "book", properties: { key: "zed" },
    next: { target: {} } } } });
  const zed = Plain.FromPlain(store)(store.registered("References.OfObject"), map({ path: {} }).set("path", nobody));
  raises(LookupError, () => layer.transaction((txn: any) => {
    const streamed = frames(items(txn.exec([new Map([["x", joined.get("x")]]), new Map([["x", zed]])],
      expression(store, older), true)));
    assert(equal(streamed.map((frame: any) => frame.errors.length > 0), [false, true])); // annotated, nothing thrown by exec
    assert(streamed[1].errors[0].subterm === "$.mutations[0].update");
  }), "no object of the store has its reference's path");
  assert(equal(people(store, layer)[1][1], ["Bob", 3n, []])); // the scope threw: every write undone
}

<a id="LAY-04"></a>
## LAY-04 · Unique clauses, the base class's, checked at a layer's commit over what it inserted: a write that leaves two entries agreeing on a clause's key refused, whole, with the message a store in memory gives; removing the entry held first, accepted

In [ ]:
{
  const [store, layer] = pair();
  const twice = (t: any): any => t.uid("twice/1").mutations([(mu: any) => mu.map((m: any) => m.symbols([(p: any) => p
    .symbols({ x: t.crm.Person, book: t.crm.Book }).spec([(_: any) => _.x.name("Bob")])]).then([(w: any) => w.update(
    (u: any) => u.apply([(p: any) => p.symbols({ x: t.crm.Person, book: t.crm.Book }).spec([
      (_: any) => _.book.contacts({ ada: (e: any) => e.contact(_.x) })])]).then([(w2: any) => w2.reduce(
    (r: any) => r.symbols([(c: any) => c.symbols(["x"])]))]))]))]); // Bob under Ada's key too; the book kept
  const refusal = (block: () => unknown): string => {
    try {
      block();
    } catch (error) {
      assert(error instanceof ValueError);
      return error.message;
    }
    throw new Error("AssertionError: expected a refusal");
  };
  const [theirs, mine] = [refusal(() => store.transaction(twice)), refusal(() => run(layer, new Frame(), twice, true))];
  assert(mine === theirs && mine.includes("breaks unique("));
  assert(layer.relations.get("crm.Holds")!.length === 2); // refused, whole
  const moved = (t: any): any => t.uid("moved/1").mutations([(mu: any) => mu.map((m: any) => m.symbols([(p: any) => p
    .symbols({ x: t.crm.Person, book: t.crm.Book }).spec([(_: any) => _.x.name("Bob")])]).then([(w: any) => w.update(
    (u: any) => u.apply([(p: any) => p.symbols({ x: t.crm.Person, book: t.crm.Book }).spec([
      (_: any) => _.book.contacts((a: any) => a.remove((e: any) => e.contact(_.x))),
      (_: any) => _.book.contacts({ robert: (e: any) => e.contact(_.x) })])]).then([(w2: any) => w2.reduce(
    (r: any) => r.symbols([(c: any) => c.symbols(["x"])]))]))]))]);
  assert(equal(paths(store, frames(run(layer, new Frame(), moved, true))), paths(store, store.transaction(moved))));
  assert(equal(layer.relations.get("crm.Holds")!.map((entry: Entry) => entry.values.get("key")).sort(), ["ada", "robert"]));
}